# Capability — Image Captioning
Generate concise or rich descriptions for any scene, optionally returning grounded regions alongside the text.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericpence/perceptron_repo/blob/main/cookbook/recipes/capabilities/captioning/captioning.ipynb)

## Install dependencies
Install the SDK and Pillow so we can preview local assets inline.

In [ ]:
!uv pip install --upgrade perceptron pillow

## Configure the Perceptron client
Set your API key once, reuse the configured client for the rest of the notebook, and resolve the captioning assets.

In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

from IPython.display import Image as IPyImage, display
from PIL import Image, ImageDraw, ImageFont

from perceptron import caption, configure
from perceptron.pointing.geometry import scale_box_to_pixels

PERCEPTRON_API_KEY = os.environ.get("PERCEPTRON_API_KEY", "<your Perceptron API key>")

configure(
    provider="perceptron",
    api_key=PERCEPTRON_API_KEY,
)

BASE_URL = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/caption/"
SUBURBAN_STREET = Path("suburban_street.webp")
SOLAR_ARRAY = Path("solar_array.webp")
SOLAR_ANNOTATED = Path("solar_array_annotated.png")

for filename, path_obj in (("suburban_street.webp", SUBURBAN_STREET), ("solar_array.webp", SOLAR_ARRAY)):
    if not path_obj.exists():
        urlretrieve(BASE_URL + filename, path_obj)


## Describe a suburban street
The simplest call requests a concise caption and returns plain text.

In [ ]:
display(IPyImage(filename=str(SUBURBAN_STREET)))
street_caption = caption(str(SUBURBAN_STREET), style="concise", expects="text")
print(street_caption.text)

## Request grounded details
Switch styles and ask for bounding boxes so downstream tooling can highlight each part of the caption.

In [ ]:
solar_caption = caption(
    str(SOLAR_ARRAY),
    style="detailed",
    expects="box",
)
print(solar_caption.text)
boxes = solar_caption.points or []
print(f"Returned {len(boxes)} grounded snippets")

In [ ]:
img = Image.open(SOLAR_ARRAY).convert("RGB")
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("arial.ttf", size=20)
except OSError:
    font = ImageFont.load_default()

if boxes:
    for idx, box in enumerate(boxes):
        scaled = scale_box_to_pixels(box, width=img.width, height=img.height)
        top_left = scaled.top_left
        bottom_right = scaled.bottom_right
        tlx, tly = int(round(top_left.x)), int(round(top_left.y))
        brx, bry = int(round(bottom_right.x)), int(round(bottom_right.y))
        draw.rectangle([tlx, tly, brx, bry], outline="orange", width=3)
        label = box.mention or f"snippet {idx + 1}"
        draw.text((tlx, max(tly - 18, 0)), label, fill="orange", font=font)
else:
    print("No grounded regions returned; set expects='box' to request them.")

img.save(SOLAR_ANNOTATED)
display(IPyImage(filename=str(SOLAR_ANNOTATED)))
print(f"Saved annotated caption overlay to {SOLAR_ANNOTATED}")


## Conclusion & next steps
- Adjust the `style` argument (`concise`, `detailed`) to match your UX.
- Keep `expects="text"` for pure narrative outputs or request grounding via `expects="box"` / `"point"`.
- Wrap this logic in a helper that iterates through folders to batch caption datasets.